In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

## Clean District Information

```district_master``` - This is a cleaned up list of school districts pulled together from 2013 & 2022-2025 years of data, keeping only the districts and fields that exist in all 5 years i.e. district ID, district name, county, region, and a couple of simple flags like whether it’s a charter so I can quickly see what each district.

***To-Dos / Notes***
- This needs a lot more cleaning...
- Re-title columns
- I will eventually need to re-clean this because rn It very bare bones
- I need to consider charter is not separated
- Double-check that matching district IDs actually represent the same real district across all years
- for all incoming data I will need to remove districts not in my district_final_cut 
- and if I go back more than 2022 if one of my districts in the final cut disappears then I need to remove them from the consideration to.
- 

i.e. 
a district is not listed for any number of reasons before 2019 but in my current list of districts from 2022-2025 if i used data back to 2014 then it will skew... actually I just loaded in 2012 and added that to the set cross refrence and it narrowd down the list a little more so im just going with those ones. 




In [2]:
# Load in District data
district_2025 = pd.read_csv(r'District Infromation\2025 District Reference.csv')
district_2024 = pd.read_csv(r'District Infromation\2024 District Reference.csv')
district_2023 = pd.read_csv(r'District Infromation\2023 District Reference Information.csv')
district_2022 = pd.read_csv(r'District Infromation\2022 District Reference Information, Accountability Rating and Special Education Determination Status.csv')
district_2013 = pd.read_csv(r'District Infromation/2013 District Reference .dat', dtype=str)

In [3]:
# Promote index 0 to headers and add year columns Year to 2024 & 2025

def promote_index_to_headers(df):
    clean_headers = df.iloc[0]
    df = df[1:]
    df.columns = clean_headers
    return df


district_25 = promote_index_to_headers(district_2025)
district_25.insert(loc=0, column="Year", value="2025")

district_24 = promote_index_to_headers(district_2024)
district_24.insert(loc=0, column="Year", value="2024")

district_25.head(1)

,Year,DISTRICT,DISTNAME,COUNTY,CNTYNAME,REGION,REGNNAME,DFLCHART,DFLALTED,D_RATING,DAD_POST,ASVAB_STATUS,OUTCOME
1,2025,001902,CAYUGA ISD,001,ANDERSON,07,REGION 07: KILGORE,N,N,B,1,,Meets Requirements


In [4]:
# Strip the leading ' in front of County, Region, District (most likely there due to excel being common for analysis and it would keep those as strings not int's)
district_2022['COUNTY'] = district_2022['COUNTY'].str[1:]
district_2022['REGION'] = district_2022['REGION'].str[1:]
district_2022['DISTRICT'] = district_2022['DISTRICT'].str[1:]

district_2023['COUNTY'] = district_2023['COUNTY'].str[1:]
district_2023['REGION'] = district_2023['REGION'].str[1:]
district_2023['DISTRICT'] = district_2023['DISTRICT'].str[1:]

In [5]:
# Adding year column to 2013
district_2013.insert(loc=0, column="Year", value="2013")
district_2022.insert(loc=0, column="Year", value="2022")
district_2023.insert(loc=0, column="Year", value="2023")

district_22 = district_2022
district_23 = district_2023

In [6]:
# This is a cleaned list of school districts I’m keeping by taking only the districts that show up in every dataset (2025, 2024, 2023, 2022, and 2013), 
# based on the assumption that if a district ID exists across all years, it’s stable enough to trust and include.

# To-do: Double-check that matching district IDs actually represent the same real district across all years
d_25 = set(district_25['DISTRICT'])
d_24 = set(district_24['DISTRICT'])
d_23 = set(district_23['DISTRICT'])
d_22 = set(district_22['DISTRICT'])
d_12 = set(district_2013['DISTRICT'])

# Creating a set of districts only in all
District_final_cut = d_25 & d_24 & d_23 & d_22 & d_12
District_final_cut
len(District_final_cut)


1162

In [7]:
# Check district lengths aren't changing  
# 2025: Original List Len: 1208 New List Len: 1208
# 2024: Original List Len: 1207 New List Len: 1207
# 2023: Original List Len: 1209 New List Len: 1209
# 2022: Original List Len: 1207 New List Len: 1207

print(f'Original List Len: {len(district_22['DISTRICT'])} New Set Len: {len(d_22)}')


Original List Len: 1207 New Set Len: 1207


In [8]:
def remove_districts(districts):
    filter = districts['DISTRICT'].isin(District_final_cut)
    filtered_district = districts[filter]
    return filtered_district

In [9]:
district_25_filtered = remove_districts(district_25)
district_24_filtered = remove_districts(district_24)
district_23_filtered = remove_districts(district_23)
district_22_filtered = remove_districts(district_22)
district_13_filtered = remove_districts(district_2013)


In [10]:
dist_25_col = set(district_25_filtered.columns)
dist_24_col = set(district_24_filtered.columns)
dist_23_col = set(district_23_filtered.columns)
dist_22_col = set(district_22_filtered.columns)
dist_13_col = set(district_2013.columns)

dist_col_in_all = dist_25_col & dist_24_col & dist_23_col & dist_22_col & dist_13_col
dist_col_in_all

{'CNTYNAME',
 'COUNTY',
 'DFLALTED',
 'DFLCHART',
 'DISTNAME',
 'DISTRICT',
 'REGION',
 'Year'}

In [11]:
# now I need to make a new data frame that is a version of all of these where all the data is the same, and I want to see what wrong with the ones that arent.
# i.e. if a cdn is around an but the district name is diff or county etc

dist_df_list = [district_25_filtered, district_24_filtered, district_23_filtered, district_22_filtered, district_13_filtered]

districts_grouped = pd.concat(
    [df.loc[:, df.columns.intersection(dist_col_in_all)] for df in dist_df_list]   
)

districts_grouped

,Year,DISTRICT,DISTNAME,COUNTY,CNTYNAME,REGION,DFLCHART,DFLALTED
1,2025,001902,CAYUGA ISD,001,ANDERSON,07,N,N
2,2025,001903,ELKHART ISD,001,ANDERSON,07,N,N
3,2025,001904,FRANKSTON ISD,001,ANDERSON,07,N,N
4,2025,001906,NECHES ISD,001,ANDERSON,07,N,N
5,2025,001907,PALESTINE ISD,001,ANDERSON,07,N,N
...,...,...,...,...,...,...,...,...
1223,2013,252902,NEWCASTLE ISD,252,YOUNG,09,N,N
1224,2013,252903,OLNEY ISD,252,YOUNG,09,N,N
1225,2013,253901,ZAPATA COUNTY ISD,253,ZAPATA,01,N,N
1226,2013,254901,CRYSTAL CITY ISD,254,ZAVALA,20,N,N


In [12]:
# Check how many unique 'charter_status' values each district has
charter_check = districts_grouped.groupby('DISTRICT')['DFLALTED'].nunique()

aea_check = districts_grouped.groupby('DISTRICT')['DFLALTED'].nunique()

# IDs where the status changed (more than 1 unique value)
# inconsistent_ids = charter_check[charter_check > 1].index  # Nothing
inconsistent_ids = aea_check[aea_check > 1].index
inconsistent_ids

Index(['015807', '015808', '101804', '101837', '101842', '101864', '193801'], dtype='str', name='DISTRICT')

In [13]:
districts_grouped.shape

districts_cleaned = districts_grouped[~districts_grouped['DISTRICT'].isin(inconsistent_ids)]

print(districts_grouped.shape, districts_cleaned.shape)


(5810, 8) (5775, 8)


In [14]:
districts_cleaned

,Year,DISTRICT,DISTNAME,COUNTY,CNTYNAME,REGION,DFLCHART,DFLALTED
1,2025,001902,CAYUGA ISD,001,ANDERSON,07,N,N
2,2025,001903,ELKHART ISD,001,ANDERSON,07,N,N
3,2025,001904,FRANKSTON ISD,001,ANDERSON,07,N,N
4,2025,001906,NECHES ISD,001,ANDERSON,07,N,N
5,2025,001907,PALESTINE ISD,001,ANDERSON,07,N,N
...,...,...,...,...,...,...,...,...
1223,2013,252902,NEWCASTLE ISD,252,YOUNG,09,N,N
1224,2013,252903,OLNEY ISD,252,YOUNG,09,N,N
1225,2013,253901,ZAPATA COUNTY ISD,253,ZAPATA,01,N,N
1226,2013,254901,CRYSTAL CITY ISD,254,ZAVALA,20,N,N


In [15]:
# using 2025 as source of truth

district_clean = districts_cleaned[districts_cleaned["Year"]=='2025'].reset_index(drop=True)


district_clean.head()

,Year,DISTRICT,DISTNAME,COUNTY,CNTYNAME,REGION,DFLCHART,DFLALTED
0,2025,001902,CAYUGA ISD,001,ANDERSON,07,N,N
1,2025,001903,ELKHART ISD,001,ANDERSON,07,N,N
2,2025,001904,FRANKSTON ISD,001,ANDERSON,07,N,N
3,2025,001906,NECHES ISD,001,ANDERSON,07,N,N
4,2025,001907,PALESTINE ISD,001,ANDERSON,07,N,N


In [16]:
# renaming columns and source of truth 

district_clean.columns = ['Year', 'District ID', 'District Name', 'County ID', 'County Name', 'Region', 'Charter Flag', 'DFLALTED']
district_clean.head(2)

,Year,District ID,District Name,County ID,County Name,Region,Charter Flag,DFLALTED
0,2025,001902,CAYUGA ISD,001,ANDERSON,07,N,N
1,2025,001903,ELKHART ISD,001,ANDERSON,07,N,N


## Clean Student & Staff Datasets

```staff_master``` - This is all staff data from 2013 through 2025 merged the same way, with only the shared columns kept so everything lines up cleanly year to year.

```student_master``` - This is all student data from 2013 through 2025 merged together, keeping only the columns that exist in every year so the data is consistent across time.

***To-Dos / Notes***
- Account for any data masking (haven't handled that yet)
- Add testing and validation throughout the process to validate each step.
- Clean up the keys table, to account for columns that are the same thing but named differently depending on the yea, handle fields that only exist in some years but are still useful.

***Useful links***
- View had data is masked: https://rptsvr1.tea.texas.gov/perfreport/tapr/2025/masking.html (change year value 2013-2025)
- 2013 data download headers explained: https://rptsvr1.tea.texas.gov/perfreport/tapr/2013/download/taprref.html

In [17]:
# Load in all staff data
StudentStaff_Info_2013 = pd.read_csv(r"Staff & Student Data\2013 Staff & Student Information.txt", dtype=str)
StudentStaff_Info_2014 = pd.read_csv(r"Staff & Student Data\2014 Staff & Student Information.dat", dtype=str)
StudentStaff_Info_2015 = pd.read_csv(r"Staff & Student Data\2015 Staff & Student Information.dat", dtype=str)
StudentStaff_Info_2016 = pd.read_csv(r"Staff & Student Data\2016 District Staff & Student Information.dat", dtype=str)
StudentStaff_Info_2017 = pd.read_csv(r"Staff & Student Data\2017 Staff & Student Information.dat", dtype=str)
StudentStaff_Info_2018 = pd.read_csv(r"Staff & Student Data\2018 Staff, Student, and Annual Graduates.dat", dtype=str)
StudentStaff_Info_2019 = pd.read_csv(r"Staff & Student Data\2019 Staff, Student, and Annual Graduates.dat", dtype=str)
StudentStaff_Info_2020 = pd.read_csv(r"Staff & Student Data\2020 Staff, Student, and Annual Graduates.dat", dtype=str)
StudentStaff_Info_2021 = pd.read_csv(r"Staff & Student Data\2021 Staff, Student, and Annual Graduates.csv", dtype=str)
StudentStaff_Info_2022 = pd.read_csv(r"Staff & Student Data\2022 Staff, Student, and Annual Graduates.csv", dtype=str)
StudentStaff_Info_2023 = pd.read_csv(r"Staff & Student Data\2023 Staff, Student, and Annual Graduates.csv", dtype=str)
Student_Info_2024 = pd.read_csv(r"Staff & Student Data\2024 District Student Information.csv", dtype=str)
Staff_Info_2024 = pd.read_csv(r"Staff & Student Data\2024 District Staff Information.csv", dtype=str)
Student_Info_2025 = pd.read_csv(r"Staff & Student Data\2025 District Student Information.csv", dtype=str)
Staff_Info_2025 = pd.read_csv(r"Staff & Student Data\2025 District Staff Information.csv", dtype=str)


### Seprate Student and Staff Data 2013 - 2023

- When I downloaded the datasets for student and staff information 2013 and 2023 the student staff data was combined. I will want to separate these later but to make managing this many tables scalable I will throw then in a dict. ```Note: I will add 2024 and 2025 separated student and staff DataFrames to a dictionary later.```
- Load in the student staff key files

Future Goal: Combining all student and staff data downloads in master keys based on all unique rows but i think a function will be quicker for me to do now so I'll merge the headers later

In [18]:
# Add student and staff 2013 and 2023 combined dataframes to a dictionary.

student_staff_dfs = {2013:StudentStaff_Info_2013, 
                      2014:StudentStaff_Info_2014, 
                      2015:StudentStaff_Info_2015, 
                      2016:StudentStaff_Info_2016, 
                      2017:StudentStaff_Info_2017,
                      2018:StudentStaff_Info_2018,
                      2019:StudentStaff_Info_2019, 
                      2020:StudentStaff_Info_2020,
                      2021:StudentStaff_Info_2021,
                      2022:StudentStaff_Info_2022,
                      2023:StudentStaff_Info_2023}
type(student_staff_dfs)

dict

In [19]:
# Loading in all the student and staff keys for 2013 to 2024

student_key_tables ={}
staff_key_tables = {}

for year in range(2013, 2024):
    # Load in files with a year variable
    student_key = pd.read_csv(rf"Staff & Student Keys\{year}_TAPR_Advanced_Data_Download_Student.csv")
    staff_key = pd.read_csv(rf"Staff & Student Keys\{year}_TAPR_Advanced_Data_Download_Staff.csv")

    student_key_tables[year] = student_key
    staff_key_tables[year] = staff_key

print(len(student_key_tables), len(staff_key_tables))


11 11


In [20]:
# For the 2024 and 2025 school years, data sets had the header a human readable name I stripped that for now and promoted index 0 which used the old id naming conversion

# Note: I'll eventually need to save those header
def PromoteFirstRow(df, year):  
    new_headers = df.iloc[0]
    new_df = df.iloc[1:]
    new_df.columns = new_headers
    new_df = new_df.reset_index(drop=True)
    key_df = pd.DataFrame(new_headers)
    key_df = key_df.reset_index()
    key_df.columns = ["LABEL", "NAME"]

    return new_df, key_df

Student_Info_2024, key_df_stu_24 = PromoteFirstRow(Student_Info_2024, 2024)
student_key_tables[2024] = key_df_stu_24

Student_Info_2025, key_df_stu_25= PromoteFirstRow(Student_Info_2025, 2025)
student_key_tables[2025] = key_df_stu_25

Staff_Info_2024, key_df_staff_24= PromoteFirstRow(Staff_Info_2024, 2024)
staff_key_tables[2024] = key_df_staff_24

Staff_Info_2025, key_df_staff_25= PromoteFirstRow(Staff_Info_2025, 2025)
staff_key_tables[2025] = key_df_staff_25


In [21]:
# CREATE: two dictionary one holding all the student data another holding all the staff data 

student_tables = {}
staff_tables = {}

for year in range(2013, 2024):
    # In the dictionary student_tables we have {year: DataFrame} i want to copy the data from the respective 


    student_tables[year] = student_staff_dfs[year][student_key_tables[year]['NAME']].copy()
    staff_tables[year] = student_staff_dfs[year][staff_key_tables[year]['NAME']].copy()

    student_tables[year].insert(loc=0, column="Year", value=year)
    staff_tables[year].insert(loc=0, column="Year", value=year)

    if year == 2023:

        
        student_tables[2024] = Student_Info_2024
        student_tables[2025] = Student_Info_2025
        student_tables[2024].insert(loc=0, column="Year", value=2024)
        student_tables[2025].insert(loc=0, column="Year", value=2025)

        staff_tables[2024] = Staff_Info_2024
        staff_tables[2025] = Staff_Info_2025
        staff_tables[2024].insert(loc=0, column="Year", value=2024)
        staff_tables[2025].insert(loc=0, column="Year", value=2025)

# Removing the ' character from districts column, likely put there for when excel was used to know the field was a string
staff_tables[2021]['DISTRICT'] = staff_tables[2021]['DISTRICT'] .str[1:]
staff_tables[2022]['DISTRICT'] = staff_tables[2022]['DISTRICT'] .str[1:]
staff_tables[2023]['DISTRICT'] = staff_tables[2023]['DISTRICT'] .str[1:]

student_tables[2021]['DISTRICT'] = student_tables[2021]['DISTRICT'] .str[1:]
student_tables[2022]['DISTRICT'] = student_tables[2022]['DISTRICT'] .str[1:]
student_tables[2023]['DISTRICT'] = student_tables[2023]['DISTRICT'] .str[1:]

In [22]:
# Create compost ID. Year + District string
for year in student_tables.keys():
    student_tables[year].insert(loc=0, column="YearDistrict ID", value=(student_tables[year]['Year'].astype(str) + student_tables[year]["DISTRICT"].astype(str)))
    staff_tables[year].insert(loc=0, column="YearDistrict ID", value = (staff_tables[year]['Year'].astype(str)  + staff_tables[year]["DISTRICT"].astype(str))) 

student_tables[2018].head(2)

,YearDistrict ID,Year,DISTRICT,DPETGEEC,DPETGEEP,DPETGPKC,DPETGPKP,DPETGKNC,DPETGKNP,DPETG01C,DPETG01P,DPETG02C,DPETG02P,DPETG03C,DPETG03P,DPETG04C,DPETG04P,DPETG05C,DPETG05P,DPETG06C,DPETG06P,DPETG07C,DPETG07P,DPETG08C,DPETG08P,DPETG09C,DPETG09P,DPETG10C,DPETG10P,DPETG11C,DPETG11P,DPETG12C,DPETG12P,DPETBLAC,DPETBLAP,DPETALLC,DPETINDC,DPETINDP,DPETASIC,DPETASIP,DPETHISC,DPETHISP,DPETPCIC,DPETPCIP,DPETTWOC,DPETTWOP,DPETWHIC,DPETWHIP,DPETDISC,DPETDISD,DPETDISP,DPETRSKC,DPETRSKP,DPETECOC,DPETECOP,DPETLEPC,DPETLEPP,DPETNEDC,DPETNEDP,DPETSAUC,DPETSAUP,DPETSBHC,DPETSBHP,DPETSINC,DPETSINP,DPETSNOC,DPETSNOP,DPETSPHC,DPETSPHP,DPETSIMC,DPETBILC,DPETBILP,DPETVOCC,DPETVOCP,DPETGIFC,DPETGIFP,DPETSPEC,DPETSPEP,DPEMALLC,DPEMALLT,DPEMALLP,DPERRA1R,DPERRA2R,DPERRA3R,DPERRA4R,DPERRA5R,DPERRA6R,DPERRA7R,DPERRA8R,DPERRA9R,DPERRAKR,DPERSA1R,DPERSA2R,DPERSA3R,DPERSA4R,DPERSA5R,DPERSA6R,DPERSA7R,DPERSA8R,DPERSA9R,DPERSAKR,DAUND17C,DAUND17D,DAUND17R
0,2018001902,2018,001902,0,0,10,1.7,28,4.9,45,7.8,41,7.1,51,8.9,51,8.9,35,6.1,45,7.8,48,8.3,53,9.2,46,8,36,6.3,45,7.8,41,7.1,20,3.5,575,0,0,5,0.9,39,6.8,0,0,26,4.5,485,84.3,3,632,0.5,226,39.3,247,43,1,0.2,328,57,-1,-1,-3,-3,58,69.9,0,0,16,19.3,83,1,0.2,160,27.8,57,9.9,83,14.4,78,579,13.5,0,2.5,0,0,3.2,0,0,0,0,0,22.2,0,14.3,0,0,0,0,0,0,33.3,0,299,0
1,2018001903,2018,001903,1,0.1,65,5.3,89,7.2,96,7.8,85,6.9,86,7,73,5.9,78,6.3,86,7,92,7.5,99,8,117,9.5,107,8.7,88,7.1,71,5.8,48,3.9,1233,3,0.2,8,0.6,137,11.1,2,0.2,49,4,986,80,5,1362,0.4,363,29.4,677,54.9,19,1.5,556,45.1,-3,-3,22,17.1,67,51.9,-1,-1,30,23.3,129,17,1.4,397,32.2,41,3.3,129,10.5,167,1192,14,5.6,2.7,0,0,0,0,2.2,1.1,2.4,13.3,0,7.1,0,0,0,0,0,0,0,18.2,0,646,0


In [23]:
# VALIDATION: Looking at unique rows, 
def valuate_year_district(df, col_1, col_2):
    x = df[col_1].nunique()
    y = df[col_2].nunique()
    diff = x/y
    return diff, x, y

for year in student_tables.keys():
    print(f"{year}: {valuate_year_district(student_tables[year], 'DISTRICT', 'Year')}")

2013: (1228.0, 1228, 1)
2014: (1227.0, 1227, 1)
2015: (1219.0, 1219, 1)
2016: (1207.0, 1207, 1)
2017: (1203.0, 1203, 1)
2018: (1200.0, 1200, 1)
2019: (1201.0, 1201, 1)
2020: (1202.0, 1202, 1)
2021: (1204.0, 1204, 1)
2022: (1207.0, 1207, 1)
2023: (1209.0, 1209, 1)
2024: (1207.0, 1207, 1)
2025: (1208.0, 1208, 1)


In [24]:
# VALIDATION: doing a basic test to see if the shapes make sense not sure why its plus one but not worries enough to check rn... i feel like it something to do with counting a header or index
for i in range(2013, 2024):    
    print(f"[{i}]: New Shape: {staff_tables[i].shape}, Expected shape: ({student_staff_dfs[i].shape[0]}, {staff_key_tables[i].shape[0]})")

[2013]: New Shape: (1228, 139), Expected shape: (1228, 137)
[2014]: New Shape: (1227, 139), Expected shape: (1227, 137)
[2015]: New Shape: (1219, 139), Expected shape: (1219, 137)
[2016]: New Shape: (1207, 139), Expected shape: (1207, 137)
[2017]: New Shape: (1203, 152), Expected shape: (1203, 150)
[2018]: New Shape: (1200, 156), Expected shape: (1200, 154)
[2019]: New Shape: (1201, 156), Expected shape: (1201, 154)
[2020]: New Shape: (1202, 117), Expected shape: (1202, 115)
[2021]: New Shape: (1204, 122), Expected shape: (1204, 120)
[2022]: New Shape: (1207, 128), Expected shape: (1207, 126)
[2023]: New Shape: (1209, 122), Expected shape: (1209, 120)


#### Current Data State (Checkpoint)

```student_tables = {}```

```staff_tables = {}```

```student_key_tables ={}```

```staff_key_tables = {}```

Note: *If comparing the data shown from this year’s report to, previous reports, use the data displayed under Membership.* [Source](https://tea.texas.gov/texas-schools/accountability/academic-accountability/performance-reporting/2020-profile-glossary.pdf)


Having difficulty understanding why the student count appears in 2023 data

In [25]:
# FILTER: Remove rows from LABEL column containing ['Percent', 'Rate', 'Ratio', '%', 'Enrollment', "Average"]

"""
staff_key_tables: Dict[df_1, df_2, ...]

"""

bad_strings = ['Percent', 'Rate', 'Ratio', '%', 'Enrollment', "Average", "Mobility", 'DAEP']
pattern = "|".join(bad_strings)

for year in range(2013, 2026):
    student_key_tables[year] = student_key_tables[year][~student_key_tables[year]["LABEL"].str.contains(pattern, na=False)]
    staff_key_tables[year] = staff_key_tables[year][~staff_key_tables[year]["LABEL"].str.contains(pattern, na=False)]

    student_key_tables[year] = student_key_tables[year].reset_index(drop=True)
    staff_key_tables[year] = staff_key_tables[year].reset_index(drop=True)

# staff_key_tables[2019]

In [26]:
# FILTER: keys table staff & student only rows whose NAME matches a pattern, producing clean indexed subsets.

"""
Data cleaning feel like sculpting clay
your change the shape, fold it, remove pars
"""
student_string = r"DPE"
staff_string = r"" 

student_24 = student_key_tables[2024]
keep_columns_student_df = student_24[student_24['NAME'].apply(str).str.contains(student_string)].copy()
keep_columns_student_df = keep_columns_student_df.reset_index(drop=True)

staff_24 = staff_key_tables[2024]
keep_columns_staff_df = staff_24[staff_24['NAME'].apply(str).str.contains(staff_string)].copy()
keep_columns_staff_df = keep_columns_staff_df.reset_index(drop=True)

print(keep_columns_student_df.shape, student_key_tables[2024].shape)

(51, 2) (66, 2)


In [27]:
# VISUAL: Summaries what was retained/removed after the filtering step above. - staff & student key tables

print("=" * 60)
print("Checkpoint: Staff & Student key tables ")
print("=" * 60)

# ── Student ────────────────────────────────────────────────────────────────
student_original = student_key_tables[2024].shape[0]
student_kept     = keep_columns_student_df.shape[0]
student_removed  = student_original - student_kept

print(f"\n  ── Student Columns DataFrame ──")
print(f"    Filter string   : '{student_string}'")
print(f"    Original rows   : {student_original}")
print(f"    Rows kept       : {student_kept}")
print(f"    Rows removed    : {student_removed}")

# ── Staff ──────────────────────────────────────────────────────────────────
staff_original   = staff_key_tables[2024].shape[0]
staff_kept       = keep_columns_staff_df.shape[0]
staff_removed    = staff_original - staff_kept

print(f"\n  ── Staff Columns DataFrame ──")
print(f"    Filter string   : '{staff_string}'")
print(f"    Original rows   : {staff_original}")
print(f"    Rows kept       : {staff_kept}")
print(f"    Rows removed    : {staff_removed}")

Checkpoint: Staff & Student key tables 

  ── Student Columns DataFrame ──
    Filter string   : 'DPE'
    Original rows   : 66
    Rows kept       : 51
    Rows removed    : 15

  ── Staff Columns DataFrame ──
    Filter string   : ''
    Original rows   : 63
    Rows kept       : 63
    Rows removed    : 0


In [28]:
# CLEAN VALUES: Remove every thing before the colon in the keys table: aka the "district 2024"

keep_columns_student_df['LABEL'] = keep_columns_student_df['LABEL'].str.split(':').str[-1].str.strip()
keep_columns_staff_df['LABEL'] = keep_columns_staff_df['LABEL'].str.split(':').str[-1].str.strip()


In [29]:
# FILTER: filter columns

id_columns = ['DISTRICT', 'Year']

def filter_columns(keys_df, dirty_df):
    key_cols = list(keys_df["NAME"])
    allowed_cols = key_cols + id_columns
    filtered_df = dirty_df.reindex(columns=allowed_cols)
    
    return filtered_df

In [30]:
# Creating long df for student and staff combined 2013-2025 data sets. 

student_long_df = []
staff_long_df = []

for year in range(2017, 2026):
    # keep columns in that I cleaned
    filtered_student_df = filter_columns(keep_columns_student_df, student_tables[year])
    filtered_staff_df = filter_columns(keep_columns_staff_df, staff_tables[year])

    student_long_df.append(filtered_student_df)
    staff_long_df.append(filtered_staff_df)

student_long_df = pd.concat(student_long_df, ignore_index=True)
staff_long_df   = pd.concat(staff_long_df, ignore_index=True)

column_to_move = student_long_df.pop('Year')
student_long_df.insert(0, 'Year', column_to_move)

column_to_move = staff_long_df.pop('Year')
staff_long_df.insert(0, 'Year', column_to_move)



In [31]:
staff_long_df['DISTRICT'].nunique()/staff_long_df['Year'].nunique()

DISTRICT    137.222222
DISTRICT    137.222222
dtype: float64

In [32]:
# FILTER: Removing duplicate "DISTRICTS" column at the end of staff_long_df
columns_count_before = staff_long_df.shape[1]

if staff_long_df.columns.duplicated().any():
    staff_long_df = staff_long_df.iloc[:, :-1] 
    print(f"Column Removed")
else: print("No Columns Removed")

columns_count_after = staff_long_df.shape[1]

print(f'Before: {columns_count_before} After: {columns_count_after}')

Column Removed
Before: 65 After: 64


In [33]:
staff_long_df

,Year,DISTRICT,DISTNAME,DPSTTOFC,DPSUTOFC,DPSSTOFC,DPSCTOFC,DPSETOFC,DPSTTOST,DPSUTOST,DPSSTOST,DPST00FC,DPST01FC,DPST06FC,DPST11FC,DPST21FC,DPST30FC,DPST00ST,DPST01ST,DPST06ST,DPST11ST,DPST21ST,DPST30ST,DPSTNOFC,DPSTBAFC,DPSTMSFC,DPSTPHFC,DPSECOFC,DPSPTOFC,DPSRTOPC,DPSRTOFC,DPSBTOPC,DPSBTOFC,DPSXTOFC,DPSATOFC,DPSOTOFC,DPSTINFC,DPSTPIFC,DPSTASFC,DPSTBLFC,DPSTHIFC,DPSTWHFC,DPSTTWFC,DPSTMAFC,DPSTFEFC,DPSTREFC,DPSTVOFC,DPSTBIFC,DPSTCOFC,DPSTGIFC,DPSTSPFC,DPSTOPFC,DPSTURNN,DPSTURND,DPSHEXPT,DPSHTENT,DPSLEXPT,DPSLTENT,DPSAMIFC,DPSTEXPT,DPSTTENT,DMSTHCNT,DEXPHCNT,DRECHCNT
0,2017,001902,NaN,52.4,6.9,4.1,2.5,13.5,2418873.4,350003.3,246627.4,1,6,7,19.9,NaN,NaN,41075,207270,265736.1,922726.7,NaN,NaN,0,48.4,4,0,0,65.9,NaN,NaN,NaN,NaN,26.2,105.5,1.9,0,0,0,2,1,49.4,0,11.9,40.5,42.5,3.3,0,0,0.2,6.4,0,4,54.4,63.9,53.1,0,0,15.9,1054.9,534.3,NaN,NaN,NaN
1,2017,001903,NaN,103,7,6,2,26.8,4628074.1,346087,384374,2,16,15,48,NaN,NaN,62683.1,570019.5,605498.2,2216976.2,NaN,NaN,0,84,19,0,0,118,NaN,NaN,NaN,NaN,34.8,179.6,0,0,0,0,2,5,96,0,21,82,82.8,4.9,0,2.6,0.7,8.8,3.2,11,100.3,69,63.9,38,8,15.5,1668.8,1052.8,NaN,NaN,NaN
2,2017,001904,NaN,59,5,4,3,19.9,2580639.3,276976,255987,2,14,10,16,NaN,NaN,60660,482843,415099,778998.4,NaN,NaN,0,42,17,0,0,71,NaN,NaN,NaN,NaN,30.1,121,0,0,0,0,4,1,53,1,15,44,46.3,5.8,0,1,0,3.7,2.2,15.1,58.1,57.9,18,28,19,14.9,1150.8,537.6,NaN,NaN,NaN
3,2017,001906,NaN,35.3,2,3,5,7,1548155.4,90675,209088,0.3,10,5,14,NaN,NaN,11600,350436.6,203323.4,648325.7,NaN,NaN,0,35.3,0,0,0,45.3,NaN,NaN,NaN,NaN,11.9,64.2,0,0,0,0,2,0,33.3,0,9,26.3,28.4,3,0,0.5,0.5,2.9,0,2.5,34.5,42,23,21,1,8,467.7,259.4,NaN,NaN,NaN
4,2017,001907,NaN,263.5,28,17,8,74.2,11827846.6,1459280.2,1249717,23.1,80.3,52.5,52.7,NaN,NaN,755207.8,3028876.4,2303218.5,2673394.2,NaN,NaN,5.3,225.8,32.4,0,0,316.6,NaN,NaN,NaN,NaN,90.2,481,0,1,1,0,24.8,25.5,207.2,4,66.6,196.9,199.8,20,4.2,15.9,0.2,23.6,0,58.9,270.3,142.8,75,162,92,118.8,3112.5,1750.8,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10836,2025,252902,NEWCASTLE ISD,24.8,0.7,2,1,10.2,1479470,49629,165535,1.8,2.8,6.6,4.8,7.9,0.9,74320,153242.9,343537.9,319969.4,560911,27488.7,1.8,15.5,7.5,0,0,28.5,0,0,0,0,5.9,44.6,0,0,0,0,0,2,22.8,0,5,19.8,18.4,2.6,0,1.2,0,2.6,0,1.8,22.5,7,3,0,0,4.8,373.4,170.2,NaN,NaN,NaN
10837,2025,252903,OLNEY ISD,73.3,7.7,6.3,1,1,4098574.3,515342.9,499474.8,3.9,22.1,13.5,17.7,9.4,6.7,152786.2,963453.4,712909,1086081.1,629750,553594.5,3.9,51.1,18.3,0,0,88.3,1,3,0,1,33,122.3,0,0,0,1,2,5.8,64.6,0,14.4,58.9,52.7,7.8,0.5,0.2,0,8.5,3.5,17.1,72.4,21,12,2,1,23,1175.8,502.7,NaN,1,NaN
10838,2025,253901,ZAPATA COUNTY ISD,240.2,47.4,14.3,11,62.7,13664248.8,3345159.2,1183970.5,36.9,58,26,75.2,37.1,7,1764700.5,2961178.4,1435489.6,4565026.7,2459849,478004.5,2,188.9,48.3,1,0,312.9,0,14,0,5,174.5,550.1,0,0,0,0,0,233.2,6,1,44,196.2,148.6,13.8,33.6,0,0,33.9,10.2,17.9,235.9,33,33,52,52,540.1,2718.4,2449,NaN,NaN,NaN
10839,2025,254901,CRYSTAL CITY ISD,107.8,22.9,13.2,7,62.8,6129788.5,1547240.3,980979.2,14,29,16.8,29,16,3,688676.8,1469279.5,935589.5,1748961.5,1074193.2,213088.1,0,95,12.8,0,0,150.9,0,8,0,1,112.7,326.4,0,0,0,0,1,99.8,7,0,35,72.8,86.4,7.1,2.8,0.2,0.7,8,2.6,17,112.4,16,16,18,12,317.4,1182.3,1058.4,NaN,NaN,NaN


In [34]:
student_long_df.columns

Index(['Year', 'DPETGEEC', 'DPETGPKC', 'DPETGKNC', 'DPETG01C', 'DPETG02C',
       'DPETG03C', 'DPETG04C', 'DPETG05C', 'DPETG06C', 'DPETG07C', 'DPETG08C',
       'DPETG09C', 'DPETG10C', 'DPETG11C', 'DPETG12C', 'DPETALLC', 'DPETSPEC',
       'DPETBILC', 'DPETVOCC', 'DPETGIFC', 'DPETLEPC', 'DPETECOC', 'DPETNEDC',
       'DPETRSKC', 'DPETINDC', 'DPETASIC', 'DPETPCIC', 'DPETTWOC', 'DPETBLAC',
       'DPETHISC', 'DPETWHIC', 'DPETMALC', 'DPETFEMC', 'DPETDSLC', 'DPET504C',
       'DPETTT1C', 'DPETHOMC', 'DPETIMMC', 'DPETMIGC', 'DPETMLCC', 'DPETGP3C',
       'DPETGP4C', 'DPETFOSC', 'DPETVHSC', 'DPETATTC', 'DPETSIMC', 'DPETSINC',
       'DPETSPHC', 'DPETSBHC', 'DPETSAUC', 'DPETSNOC', 'DISTRICT'],
      dtype='str')

In [35]:
# CREATE: Year & Campus ID column
student_long_df.insert(loc=0, column="YearDistrict ID", value=(student_long_df['Year'].astype(str) + student_long_df["DISTRICT"].astype(str)))
staff_long_df.insert(loc=0, column="YearDistrict ID", value = (staff_long_df['Year'].astype(str)  + staff_long_df["DISTRICT"].astype(str))) 

# student_long_df

In [36]:
# FILTER: Only keep district in district_clean dataset
allowed_district = district_clean['District ID']

unique_districts_before = student_long_df['DISTRICT'].nunique()
unique_year_before = student_long_df['Year'].nunique()

def filter_districts(df):
    df = df[df['DISTRICT'].isin(allowed_district)]
    return df

type(allowed_district[0])
#student_long_df[student_long_df["DISTRICT"]].isin(allowed_district)
#student_long_df
student_long_df = filter_districts(student_long_df)
staff_long_df = filter_districts(staff_long_df)

unique_districts_after= student_long_df['DISTRICT'].nunique()
unique_rows_after = student_long_df['YearDistrict ID'].nunique()
unique_year_after= student_long_df['Year'].nunique()



print(unique_districts_before, unique_districts_after, len(allowed_district), unique_rows_after/unique_year_after)

1235 1155 1155 1155.0


In [37]:
# Replace ID headers with human readable names
def replace_col_headers(keys_df, long_df):
    label_to_name = (keys_df.set_index("NAME")["LABEL"].to_dict())
    long_df.rename(columns=label_to_name, inplace=True)
    return long_df

student_long_df = replace_col_headers(keep_columns_student_df, student_long_df)
staff_long_df = replace_col_headers(keep_columns_staff_df, staff_long_df)

In [38]:
# manual header cleaning
district_col_move = student_long_df.pop('DISTRICT')
student_long_df.insert(1, "DISTRICT", district_col_move)
student_long_df.head()

staff_long_df.rename(columns={'6 Digit County District Number': 'DISTRICT'}, inplace=True)

In [39]:
print(valuate_year_district(staff_long_df, 'YearDistrict ID', 'Year'))


(1155.0, 10395, 9)


In [40]:
student_long_df

,YearDistrict ID,DISTRICT,Year,EE Count,PK Count,KG Count,01 Count,02 Count,03 Count,04 Count,05 Count,06 Count,07 Count,08 Count,09 Count,10 Count,11 Count,12 Count,All Students Count,Special Ed Count,Bilingual/ESL Count,Career & Technical Education Count,Gifted & Talented Count,EB/EL Count,Econ Disadv Count,Non-Educationally Disadv Count,At Risk Count,American Indian Count,Asian Count,Pacific Islander Count,Two or More Races Count,African American Count,Hispanic Count,White Count,Male Count,Female Count,Dyslexia Count,Section 504 Count,Title I Count,Homeless Count,Immigrant Count,Migrant Count,Military-Connected Count,PK Count Ages 3 and Under Count,PK Count Ages 4 and Over Count,Foster Care Count,Career & Technical (9-12) Education Count,2022-23 Attrition All Students Count,Total Students with Disabilities Count,Intellectual Disabilities Count,Physical Disabilities Count,Behavioral Disabilities Count,Autism Count,Non-Categorical Early Childhood Count
0,2017001902,001902,2017,1,0,44,48,46,47,40,41,42,50,44,37,48,39,49,576,82,3,150,55,3,192,384,203,0,3,0,25,27,42,479,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82,57,15,7,-1,-1
1,2017001903,001903,2017,4,56,102,82,93,73,85,84,88,94,112,110,104,91,89,1267,139,19,299,39,19,673,594,453,1,11,2,38,67,137,1011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139,75,35,19,-3,-1
2,2017001904,001904,2017,2,27,57,49,73,65,62,59,77,68,53,66,65,68,55,846,75,18,229,59,18,446,400,356,6,6,0,28,76,82,648,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75,46,8,12,-3,-1
3,2017001906,001906,2017,0,16,34,22,28,21,29,31,33,31,24,39,26,24,19,377,37,6,118,24,6,172,205,137,1,1,0,10,28,51,286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37,30,-1,-1,-1,-1
4,2017001907,001907,2017,4,191,268,248,266,251,269,252,239,252,241,274,250,230,218,3453,303,516,1222,93,557,2587,866,1980,10,28,4,106,949,1382,974,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,303,139,56,64,39,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10836,2025252902,252902,2025,0,19,13,14,11,14,20,14,11,15,16,18,23,23,14,225,42,4,73,3,4,140,85,64,0,0,0,7,1,32,185,117,108,28,21,225,0,0,0,10,1,18,1,57,26,42,22,6,9,5,0
10837,2025252903,252903,2025,0,55,50,44,56,58,44,60,45,48,46,46,57,43,37,689,145,98,202,27,98,383,306,338,1,3,0,26,14,256,389,367,322,50,72,504,18,30,0,1,16,39,1,137,80,145,52,48,-3,25,-1
10838,2025253901,253901,2025,12,172,187,216,216,207,231,246,235,245,250,253,275,281,256,3282,566,1072,940,233,981,2917,365,1962,0,2,0,2,2,3268,8,1709,1573,253,237,3282,219,26,40,15,0,172,6,939,158,566,331,81,49,94,11
10839,2025254901,254901,2025,1,123,102,107,92,101,123,127,127,119,109,127,123,126,128,1635,298,40,575,127,41,1329,306,1077,0,0,0,0,7,1610,18,842,793,152,144,1635,85,2,12,5,53,70,4,431,95,298,148,50,50,45,5


In [41]:
# CLEAN: Handle masked data; dropping any masked values and replacing with nan ["n/a", '-1', '-2', '-3', "•"]
student_long_df = student_long_df.replace(["n/a", '-1', '-2', '-3', "•"], np.nan)
staff_long_df = staff_long_df.replace(["n/a", '-1', '-2', '-3', "•"], np.nan)

In [42]:
# filter down dataframes to 2017 - 2025
student_analysis_df = student_long_df[student_long_df['Year'] >= 2017]
staff_analysis_df = staff_long_df[staff_long_df['Year'] >= 2017]

In [43]:
staff_long_df['DISTRICT'].nunique()/staff_long_df["Year"].nunique()

128.33333333333334

In [44]:
# Add cleaned District descriptive attributes

district_columns_to_carry_over = district_clean[["District ID", 'District Name', 'County ID', 'County Name', 'Region', 'Charter Flag', 'DFLALTED']]


def add_district_dim(df):
    df = df.merge(
        district_columns_to_carry_over,
        left_on="DISTRICT",
        right_on="District ID",
        how="left"
        
    )
    df = df.drop(columns=["District ID"])

    return df

student_analysis_df = add_district_dim(student_analysis_df)
staff_analysis_df = add_district_dim(staff_analysis_df)

In [45]:
# Change Data Type: Converting the numeric values back to numbers from strings

string_cols = [
    "Year",
    "DISTRICT",
    "District Name",
    "County ID",
    "County Name",
    "Region",
    "Charter Flag",
    "DFLALTED",
]

def coerce_numeric_except(df, string_cols):
    df = df.copy()

    numeric_cols = df.columns.difference(string_cols)

    df[numeric_cols] = df[numeric_cols].apply(
        pd.to_numeric, errors="coerce"
    )

    return df

student_analysis_df = coerce_numeric_except(
    student_analysis_df,
    string_cols
)

staff_analysis_df = coerce_numeric_except(
    staff_analysis_df,
    string_cols
)

In [46]:
staff_analysis_df.shape[0]/staff_analysis_df["Year"].nunique()

1155.0

In [47]:
# FILTER: REMOVE TINY DISTRICTS AND VALIDATE

DISTRICT_SIZE_THRESHOLD = 30  # Minimum "All Students Count" required

# Identify districts to remove
tiny_districts = (
    student_analysis_df[student_analysis_df["All Students Count"] <= DISTRICT_SIZE_THRESHOLD]["DISTRICT"]
    .unique()
)

years_count = student_analysis_df["Year"].nunique()
expected_removed = len(tiny_districts)
expected_rows_removed = len(tiny_districts) * years_count


# Capture shapes BEFORE filter
districts_before = (
    student_analysis_df
    .groupby("Year")["DISTRICT"]
    .nunique()
    .rename("districts_before")
)

student_shape_before = student_analysis_df.shape
staff_shape_before = staff_analysis_df.shape

# Apply filtering
student_analysis_df = (
    student_analysis_df[~student_analysis_df["DISTRICT"].isin(tiny_districts)]
    .reset_index(drop=True)
)

staff_analysis_df = (
    staff_analysis_df[~staff_analysis_df["DISTRICT"].isin(tiny_districts)]
    .reset_index(drop=True)
)

# Capture shapes AFTER filter
districts_after = (
    student_analysis_df
    .groupby("Year")["DISTRICT"]
    .nunique()
    .rename("districts_after")
)

student_shape_after = student_analysis_df.shape
staff_shape_after = staff_analysis_df.shape

# ============================================================
# VALIDATION
# ============================================================

# ----- Row validation -----

student_rows_removed = student_shape_before[0] - student_shape_after[0]
staff_rows_removed = staff_shape_before[0] - staff_shape_after[0]

print("\nFILTER VALIDATION")
print("--------------------------------------------------")

print("\nDISTRICT FILTER")
print(f"Tiny districts removed : {len(tiny_districts)}")
print(f"Years in dataset       : {years_count}")
print(f"Expected rows removed  : {expected_rows_removed}")

print("\nROW COUNTS")

print("Students")
print(f"  Before               : {student_shape_before[0]}")
print(f"  After                : {student_shape_after[0]}")
print(f"  Rows removed         : {student_rows_removed}")

print("\nStaff")
print(f"  Before               : {staff_shape_before[0]}")
print(f"  After                : {staff_shape_after[0]}")
print(f"  Rows removed         : {staff_rows_removed}")

print("\nROW VALIDATION")
print(f"Students valid         : {student_rows_removed == expected_rows_removed}")
print(f"Staff valid            : {staff_rows_removed == expected_rows_removed}")


# ----- District validation by year -----

validation = pd.concat([districts_before, districts_after], axis=1)

validation["removed"] = validation["districts_before"] - validation["districts_after"]
validation["valid"] = validation["removed"] == len(tiny_districts)

print("\nDISTRICT COUNT VALIDATION (BY YEAR)")
print("--------------------------------------------------")
print(validation)


FILTER VALIDATION
--------------------------------------------------

DISTRICT FILTER
Tiny districts removed : 7
Years in dataset       : 9
Expected rows removed  : 63

ROW COUNTS
Students
  Before               : 10395
  After                : 10332
  Rows removed         : 63

Staff
  Before               : 10395
  After                : 10332
  Rows removed         : 63

ROW VALIDATION
Students valid         : True
Staff valid            : True

DISTRICT COUNT VALIDATION (BY YEAR)
--------------------------------------------------
      districts_before  districts_after  removed  valid
Year                                                   
2017              1155             1148        7   True
2018              1155             1148        7   True
2019              1155             1148        7   True
2020              1155             1148        7   True
2021              1155             1148        7   True
2022              1155             1148        7   True
2023       

In [48]:
# Export files to Clean Datasets
student_analysis_df.to_csv(r'Clean Datasets/student_analysis_df.csv', index=False)
staff_analysis_df.to_csv(r'Clean Datasets/staff_analysis_df.csv', index=False)


In [49]:
student_analysis_df.dtypes

YearDistrict ID                                int64
DISTRICT                                         str
Year                                           int64
EE Count                                       int64
PK Count                                       int64
KG Count                                       int64
01 Count                                       int64
02 Count                                       int64
03 Count                                       int64
04 Count                                       int64
05 Count                                       int64
06 Count                                       int64
07 Count                                       int64
08 Count                                       int64
09 Count                                       int64
10 Count                                       int64
11 Count                                       int64
12 Count                                       int64
All Students Count                            

### Financial Data
Investiage all data and figure out what to pull

[Financial Accountability System Resource Guide](https://tea.texas.gov/finance-and-grants/financial-compliance/financial-accountability-system-resource-guide)

[PEIMS Single File Financial Data Downloads](https://tea.texas.gov/finance-and-grants/state-funding/state-funding-reports-and-data/peims-single-file-financial-data-downloads)

[Object Codes](http://ritter.tea.state.tx.us/peims/standards/weds/index.html?c159)

[Another link](https://tea.texas.gov/finance-and-grants/state-funding/state-funding-reports-and-data)

## Cleaning Benchmarking data

